# Checking the Generated BTW 2025 Constituency JSON Files

This notebook follows the same role as the 2021 validation notebook: it reads the finished JSON files rather than relying on in-memory objects from the preparation run.

The focus is on file size, record structure, demographic domains, duplicate rows, coverage, example constituencies, nationwide percentage summaries and an independent reconstruction of the exact polling-district source totals.

Remember that the detailed demographic cells are modelled state-profile distributions. Passing these checks means the generated files respect the documented structure and margins; it does not turn the modelled cells into observed individual ballots.

## 0. Working directory and files

In [ ]:
from pathlib import Path
import math
import sys

import pandas as pd
from IPython.display import display


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("The notebook must run inside the repository folder.")


ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))

DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw25_wbz_ergebnisse.csv"
GENERATED_DIRECTORY = ROOT / "scripts/data/generated/btw2025"
FIRST_VOTES_JSON = GENERATED_DIRECTORY / "first_votes.json"
SECOND_VOTES_JSON = GENERATED_DIRECTORY / "second_votes.json"

from scripts.election_data.btw2025 import (
    BTW2025_AGE_GROUPS,
    read_polling_district_csv,
    reshape_polling_district_votes,
)
from scripts.election_data.notebook_steps import (
    aggregate_to_constituencies,
    inspect_district_rows,
    normalize_district_rows,
    select_usable_district_rows,
)

FIRST_VOTES_JSON, SECOND_VOTES_JSON

## 1. Files and sizes

First confirm that both generated artifacts exist and inspect their size before loading them. This is useful because an unexpectedly tiny or very large file can reveal a failed preparation run immediately.

In [ ]:
for path in (FIRST_VOTES_JSON, SECOND_VOTES_JSON):
    if not path.is_file():
        raise FileNotFoundError(f"File missing: {path}")

file_sizes = pd.DataFrame({
    "file": [FIRST_VOTES_JSON.name, SECOND_VOTES_JSON.name],
    "MiB": [FIRST_VOTES_JSON.stat().st_size / 1024**2, SECOND_VOTES_JSON.stat().st_size / 1024**2],
})

display(file_sizes)
print(f"Combined: {file_sizes['MiB'].sum():.2f} MiB")

## 2. Read the JSON and inspect the first rows

These are the finished records exactly as pandas reads them from disk. No source CSV or preparation object is involved in this step.

In [ ]:
first_votes = pd.read_json(FIRST_VOTES_JSON)
second_votes = pd.read_json(SECOND_VOTES_JSON)

print(f"First-vote rows: {len(first_votes):,}")
print(f"Second-vote rows: {len(second_votes):,}")
display(first_votes.head(20))
display(second_votes.head(20))

## 3. Expected columns and value ranges

The record fields are shared with the 2021 `VoteEntry` structure, but the permitted 2025 age-group values are election-specific. `pandas` may deserialize `voteType` as integers even though the JSON contract writes string literals, so the check compares their string representation.

The internal value `gender="m"` represents the source category `m|d|o` for this election.

In [ ]:
expected_columns = {
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
    "votes",
}
allowed_genders = {"m", "w"}
allowed_age_groups = set(BTW2025_AGE_GROUPS)
allowed_methods = {"postal", "in-person"}


def check_vote_frame(frame: pd.DataFrame, label: str, expected_vote_type: int) -> None:
    missing = expected_columns - set(frame.columns)
    unexpected = set(frame.columns) - expected_columns

    print(label, {"missing": sorted(missing), "unexpected": sorted(unexpected)})
    assert not missing
    assert not unexpected

    assert frame["districtId"].notna().all()
    assert (frame["districtId"] > 0).all()
    assert frame["state"].notna().all()
    assert frame["party"].notna().all()
    assert set(frame["gender"].unique()) <= allowed_genders
    assert set(frame["ageGroup"].unique()) <= allowed_age_groups
    assert set(frame["electionMethod"].unique()) <= allowed_methods
    assert set(frame["voteType"].astype(str).unique()) == {str(expected_vote_type)}
    assert frame["votes"].map(math.isfinite).all()
    assert (frame["votes"] >= 0).all()


check_vote_frame(first_votes, "first_votes", 1)
check_vote_frame(second_votes, "second_votes", 2)

print("Observed age groups:", sorted(set(first_votes["ageGroup"]) | set(second_votes["ageGroup"])))
print('For BTW 2025, gender="m" represents the published combined category "m|d|o".')

## 4. Look for duplicate detail rows

A detail row is uniquely identified by every field except `votes`. A duplicate means that the same constituency/demographic/party/method cell was written more than once.

In [ ]:
detail_key = [
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
]

duplicate_summary = pd.DataFrame({
    "file": ["first_votes.json", "second_votes.json"],
    "duplicate rows": [
        int(first_votes.duplicated(detail_key).sum()),
        int(second_votes.duplicated(detail_key).sum()),
    ],
})

display(duplicate_summary)
assert duplicate_summary["duplicate rows"].eq(0).all()

## 5. Inspect coverage

This overview makes missing constituencies, states, parties, gender categories, age groups or election methods visible without assuming that every party must occur in every constituency.

In [ ]:
def coverage(frame: pd.DataFrame, label: str) -> dict[str, object]:
    return {
        "file": label,
        "rows": len(frame),
        "constituencies": frame["districtId"].nunique(),
        "states": frame["state"].nunique(),
        "parties/categories": frame["party"].nunique(),
        "genders": ", ".join(sorted(frame["gender"].unique())),
        "age groups": ", ".join(sorted(frame["ageGroup"].unique())),
        "methods": ", ".join(sorted(frame["electionMethod"].unique())),
    }

coverage_table = pd.DataFrame([
    coverage(first_votes, "first_votes.json"),
    coverage(second_votes, "second_votes.json"),
])
display(coverage_table)

first_districts = set(first_votes["districtId"].unique())
second_districts = set(second_votes["districtId"].unique())
print("Only in first votes:", sorted(first_districts - second_districts))
print("Only in second votes:", sorted(second_districts - first_districts))

## 6. Reconstruct one constituency

Sum the detailed rows back over gender, age and election method. This produces ordinary party totals again and gives a readable example of what the large JSON actually represents.

In [ ]:
sample_district = int(min(first_votes["districtId"].min(), second_votes["districtId"].min()))

sample_first = (
    first_votes[first_votes["districtId"] == sample_district]
    .groupby("party", as_index=False)["votes"].sum()
    .sort_values("votes", ascending=False)
)
sample_second = (
    second_votes[second_votes["districtId"] == sample_district]
    .groupby("party", as_index=False)["votes"].sum()
    .sort_values("votes", ascending=False)
)

print(f"Constituency {sample_district}: first votes")
display(sample_first.head(20))
print(f"Constituency {sample_district}: second votes")
display(sample_second.head(20))

## 7. Nationwide election results in percent

The first table shows nationwide first- and second-vote percentages after all demographic cells, constituencies and methods have been summed back together. The following tables show the party distribution by the 2025 age groups and by the two published/internal gender categories.

In [ ]:
AGE_GROUP_ORDER = list(BTW2025_AGE_GROUPS)
GENDER_ORDER = ["m", "w"]


def nationwide_party_percentages(first_frame: pd.DataFrame, second_frame: pd.DataFrame) -> pd.DataFrame:
    first_totals = first_frame.groupby("party")["votes"].sum()
    second_totals = second_frame.groupby("party")["votes"].sum()
    result = pd.concat([
        (first_totals / first_totals.sum() * 100).rename("First vote (%)"),
        (second_totals / second_totals.sum() * 100).rename("Second vote (%)"),
    ], axis=1).fillna(0.0)
    return result.sort_values(["Second vote (%)", "First vote (%)"], ascending=False)


def demographic_percentage_table(frame: pd.DataFrame, group_column: str, party_limit: int = 12) -> pd.DataFrame:
    nationwide = frame.groupby("party")["votes"].sum().sort_values(ascending=False)
    display_parties = nationwide.head(party_limit).index.tolist()
    data = frame[[group_column, "party", "votes"]].copy()
    data["displayParty"] = data["party"].where(data["party"].isin(display_parties), "Other parties")
    grouped = data.groupby([group_column, "displayParty"])["votes"].sum()
    percentages = grouped / grouped.groupby(level=0).sum() * 100
    return percentages.unstack("displayParty").fillna(0.0)

nationwide_percentages = nationwide_party_percentages(first_votes, second_votes)
display(nationwide_percentages)

In [ ]:
print("Second-vote percentages by age group")
second_by_age = demographic_percentage_table(second_votes, "ageGroup").reindex(AGE_GROUP_ORDER)
display(second_by_age)

print("Second-vote percentages by gender category")
second_by_gender = demographic_percentage_table(second_votes, "gender").reindex(GENDER_ORDER)
display(second_by_gender)

print('Reminder: the row "m" is the published 2025 category m|d|o, not exclusively male.')

## 8. Independently reconstruct the official constituency/method totals

Finally, re-read the polling-district CSV and rebuild the official fixed margins from scratch. The generated JSON is then aggregated with the same key and compared group by group.

This is intentionally separate from the preparation notebook: a mismatch here means the finished artifacts no longer conserve the official `constituency × state × party × vote type × election method` totals.

In [ ]:
raw_districts = read_polling_district_csv(DISTRICT_RESULTS_CSV)
diagnostics = inspect_district_rows(raw_districts)
display(diagnostics["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows"))
usable_districts = select_usable_district_rows(raw_districts, diagnostics)
normalized_districts = normalize_district_rows(usable_districts)

official_totals = pd.concat([
    aggregate_to_constituencies(reshape_polling_district_votes(normalized_districts, vote_type="1")),
    aggregate_to_constituencies(reshape_polling_district_votes(normalized_districts, vote_type="2")),
], ignore_index=True)

generated = pd.concat([first_votes, second_votes], ignore_index=True)
generated["voteType"] = generated["voteType"].astype(str)
source_keys = ["districtId", "state", "party", "voteType", "electionMethod"]
generated_totals = generated.groupby(source_keys, as_index=False)["votes"].sum()

comparison = official_totals.merge(
    generated_totals,
    on=source_keys,
    how="outer",
    suffixes=("Official", "Generated"),
).fillna(0.0)
comparison["absoluteError"] = (comparison["votesOfficial"] - comparison["votesGenerated"]).abs()

display(comparison.sort_values("absoluteError", ascending=False).head(20))
print("Groups compared:", len(comparison))
print("Maximum absolute error:", comparison["absoluteError"].max())
assert comparison["absoluteError"].max() <= 1e-6

If every assertion passes, the generated BTW 2025 files have the expected `VoteEntry` shape, contain no duplicate detail keys, use only the documented 2025 demographic domains, and preserve every official constituency/party/vote-type/postal-or-in-person source total within the numerical tolerance.